In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Load data
data = pd.read_csv('data/train.csv')
# print(data.head())    # Check basic information of the dataset
# print(data.info())    # Check for missing values in the dataset

# Preprocessing
X = data.iloc[:, 1:-1]
y = data.iloc[:, -1]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=22, stratify=y)

# Feature engineering (Standardization)
transfer = StandardScaler()
x_train = transfer.fit_transform(x_train)
x_test = transfer.transform(x_test)

x_train = pd.DataFrame(x_train, columns=X.columns)
x_test = pd.DataFrame(x_test, columns=X.columns)

print('Predicting 5G users using various models: KNN, Logistic Regression, Decision Tree, Random Forest, GBDT.')
print('Data loading completed. Starting model training and prediction.')
estimator_prob = {}

# 1. KNN
start = time.time()
print("\n1. KNN Classification")
estimator = KNeighborsClassifier()
param_grid = {'n_neighbors': [7, 9, 11]}
estimator = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, n_jobs=-1, scoring='roc_auc')
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['KNN'] = prob
print(f"KNN training finished! Time: {time.time()-start:.2f}s")
print('Best params:', estimator.best_params_)
print('Best average score:', estimator.best_score_)

# 2. Logistic Regression
start = time.time()
print("\n2. Logistic Regression")
estimator = LogisticRegression(random_state=22)
param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}
estimator = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, n_jobs=-1, scoring='roc_auc')
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['Logistic Regression'] = prob
print(f"LogisticRegression training finished! Time: {time.time()-start:.2f}s")
print('Best params:', estimator.best_params_)
print('Best average score:', estimator.best_score_)

# 3. Decision Tree
start = time.time()
print('\n3. Decision Tree')
estimator = DecisionTreeClassifier(random_state=22)
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 8, 12],
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [10, 20]
}
estimator = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, n_jobs=-1, scoring='roc_auc')
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['Decision Tree'] = prob
print(f"DecisionTree training finished! Time: {time.time()-start:.2f}s")
print('Best params:', estimator.best_params_)
print('Best average score:', estimator.best_score_)

# 4. Random Forest (LightGBM rf mode)
start = time.time()
print('\n4. Random Forest')
estimator = lgb.LGBMClassifier(
    boosting_type='rf',
    bagging_fraction=0.8,
    bagging_freq=1,
    class_weight='balanced',
    random_state=22,
    n_jobs=-1, verbose=-1
)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10],
}
estimator = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, n_jobs=-1, scoring='roc_auc')
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['Random Forest'] = prob
print(f"RandomForest training finished! Time: {time.time()-start:.2f}s")
print('Best params:', estimator.best_params_)
print('Best average score:', estimator.best_score_)

# 5. GBDT
start = time.time()
print('\n5. Gradient Boosting Decision Tree (GBDT)')
estimator = lgb.LGBMClassifier(
    class_weight='balanced',
    random_state=22,
    n_jobs=-1, verbose=-1
)
param_grid = {
    'learning_rate': [0.05, 0.1],
    'num_leaves': [15, 31, 63],
}
estimator = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, n_jobs=-1, scoring='roc_auc')
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['GBDT'] = prob
print(f"GBDT training finished! Time: {time.time()-start:.2f}s")
print('Best params:', estimator.best_params_)
print('Best average score:', estimator.best_score_)

# 6. SVM
print('\n6. Support Vector Machine (SVM)')
start = time.time()
estimator = LinearSVC(C=1.0, dual=False, random_state=22)
estimator = CalibratedClassifierCV(estimator=estimator, cv=5)
estimator.fit(x_train, y_train)
prob = estimator.predict_proba(x_test)[:, 1]
estimator_prob['SVM'] = prob
print(f"SVM training finished! Time: {time.time()-start:.2f}s")
print(f"SVM AUC score: {roc_auc_score(y_test, prob):.4f}\n")

# Visualization: ROC Curve
plt.figure(figsize=(10, 8), dpi=150)
for name, prob in estimator_prob.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, prob):.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve Comparison -- 5G User Prediction')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()